<a href="https://colab.research.google.com/github/abpanic/BrightMind-AI/blob/main/Chatterbox_Turbo_Australian_Male_Colab_Drive_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Australian Male — Chatterbox **Turbo** · Google Colab GPU · Google Drive · DOCX → MP3

This is the Colab/Drive version of the working local notebook. It **always uses Turbo, not Nano**, and uses your own uploaded WAV voice reference, voice preview, paragraph/tag parsing, and MP3 batch processing.

**First:** In Colab, select **Runtime → Change runtime type → T4 GPU** (or another GPU). Run cells top to bottom and grant Google Drive access. Upload your DOCX stories to `MyDrive/Chatterbox_Aussie_Turbo/input_docx`. Output MP3s and previews are written directly to your Drive, surviving Colab session resets. By default, only a voice preview runs; enable the DOCX batch after listening.

**Important:** Google Colab sessions are temporary, and free GPU availability varies. Re-run installation and model loading after a disconnect. Drive files persist. Upload a WAV you have the right to use. It is saved to Google Drive and reused in future sessions.

**Colab dependency note:** On an existing runtime, run install cell 1, then select **Runtime → Restart session** and continue from cell 2. A restart clears imports from an incompatible earlier torch/torchvision installation. The `perth` `pkg_resources` warning is nonfatal.


In [1]:
# 1 — Install compatible Chatterbox Turbo / PyTorch packages (once per fresh Colab runtime).
# Chatterbox 0.1.7 pins torch/torchaudio 2.6.0; torchvision 0.21.0 matches that pair.
# Installing torchvision explicitly fixes: RuntimeError: operator torchvision::nms does not exist.
%pip -q install "chatterbox-tts==0.1.7" python-docx requests soundfile imageio-ffmpeg
%pip -q install --force-reinstall --no-deps "torchvision==0.21.0"
print("Installed Chatterbox Turbo and matched torchvision 0.21.0.")
print("IMPORTANT: If this runtime already failed imports or had packages installed, select Runtime > Restart session NOW, then run cells 2 onward. Do not rerun cell 1 after restart.")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires starlette<2,>=1.3.1, but you have starlette 0.52.1 which is incompatible.
torchvision 0.26.0+cu130 requires torch==2.11.0, but you have torch 2.6.0 which is incompatible.
python-fasthtml 0.14.14 requires starlette>=1.0.1, but you have starlette 0.52.1 which is incompatible.
Installed Chatterbox Turbo and matched torchvision 0.21.0.
IMPORTANT: If this runtime already failed imports or had packages installed, select Runtime > Restart session NOW, then run cells 2 onward. Do not rerun cell 1 after restart.


In [15]:
# 2 — Mount Google Drive and configure Turbo run.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os

# All inputs, reference voice, generated MP3s and reports remain in your Drive.
BASE_DIR = Path('/content/drive/MyDrive/Chatterbox_Aussie_Turbo')
DOCX_DIR = BASE_DIR / 'input_docx'
OUT_DIR = BASE_DIR / 'output_mp3_aussie_male'
VOICE_DIR = BASE_DIR / 'voices'
REPORT_DIR = BASE_DIR / 'reports_aussie_male'
for folder in (BASE_DIR, DOCX_DIR, OUT_DIR, VOICE_DIR, REPORT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# TURBO ONLY. The model is loaded on the GPU in cell 5.
RUN_VOICE_PREVIEW = True
RUN_DOCX_BATCH = True     # Set True after checking the voice preview
RUN_LIMIT = 1             # One DOCX for the first test; None = all DOCX files
OVERWRITE_EXISTING = False

REFERENCE_WAV = VOICE_DIR / 'my_voice_reference.wav'  # Persistent copy of your uploaded WAV
PREVIEW_TEXT = (
    "Right, here's a pub story. The manager called from the very traffic jam "
    "that stopped deliveries! [chuckle] You couldn't make this up."
)
MAX_CHARS = 250
PARAGRAPH_PAUSE_SECONDS = 0.24
CHUNK_PAUSE_SECONDS = 0.10
MP3_BITRATE = '192k'

print('Drive workspace:',BASE_DIR)
print('Narrator model: Chatterbox Turbo (not Nano)')
print('DOCX input:',DOCX_DIR)
print('MP3 output:',OUT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive workspace: /content/drive/MyDrive/Chatterbox_Aussie_Turbo
Narrator model: Chatterbox Turbo (not Nano)
DOCX input: /content/drive/MyDrive/Chatterbox_Aussie_Turbo/input_docx
MP3 output: /content/drive/MyDrive/Chatterbox_Aussie_Turbo/output_mp3_aussie_male


In [16]:
# 3 — Verify CUDA, torch/torchvision ABI compatibility, and MP3 encoder.
import sys, importlib.metadata
import torch
import torchvision
import torchaudio
import soundfile as sf, imageio_ffmpeg
from torchvision.ops import nms

print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__, '| torchvision:', torchvision.__version__, '| torchaudio:', torchaudio.__version__)
if torch.__version__.split('+')[0] != '2.6.0' or torchvision.__version__.split('+')[0] != '0.21.0':
    raise RuntimeError('PyTorch mismatch. Restart session; rerun installation cell 1; restart again, then run cells 2 onward.')
# Check native NMS operator really exists, rather than waiting for a model import traceback.
try:
    from torch import tensor
    nms(torch.tensor([[0.,0.,1.,1.]]),torch.tensor([0.9]),0.5)
except Exception as exc:
    raise RuntimeError('torchvision native operators are broken. Start a fresh Colab runtime and rerun installation cell 1, then restart session.') from exc
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU detected. In Colab select Runtime > Change runtime type > T4 GPU, reconnect and rerun.')
DEVICE = 'cuda'
FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
print('GPU:', torch.cuda.get_device_name(0))
print('FFmpeg:', FFMPEG)
print('Chatterbox:', importlib.metadata.version('chatterbox-tts'))
print('PyTorch/torchvision native operator check: PASSED')


Python: 3.13.16
Torch: 2.6.0+cu124 | torchvision: 0.21.0+cu124 | torchaudio: 2.6.0+cu124
GPU: Tesla T4
FFmpeg: /usr/local/lib/python3.13/dist-packages/imageio_ffmpeg/binaries/ffmpeg-linux-x86_64-v7.0.2
Chatterbox: 0.1.7
PyTorch/torchvision native operator check: PASSED


In [17]:
# 4 — Upload your own WAV reference (only once; the copy is saved in Drive).
# On the first run Colab opens a file picker. Later runs reuse the saved WAV.
from google.colab import files
from IPython.display import Audio, display
import soundfile as sf
import shutil
from pathlib import Path

if not REFERENCE_WAV.is_file():
    print('Please upload your WAV voice reference (a clean, single-speaker recording).')
    uploaded = files.upload()
    wav_files = [name for name in uploaded if name.lower().endswith('.wav')]
    if len(wav_files) != 1 or len(uploaded) != 1:
        raise ValueError('Please upload exactly one .wav file, then rerun this cell.')
    source_wav = Path(wav_files[0])
    try:
        sf.info(str(source_wav))
    except Exception as exc:
        raise ValueError('Uploaded file is not a readable WAV recording.') from exc
    shutil.copyfile(source_wav, REFERENCE_WAV)
    print('Saved reference to Google Drive:', REFERENCE_WAV)
else:
    print('Using existing voice reference from Google Drive:', REFERENCE_WAV)

ref_info = sf.info(str(REFERENCE_WAV))
if ref_info.duration <= 5.0:
    raise ValueError('The WAV reference should be longer than 5 seconds.')
print(f'Voice reference: {ref_info.duration:.1f}s, {ref_info.samplerate} Hz, {ref_info.channels} channel(s)')
display(Audio(filename=str(REFERENCE_WAV)))


Using existing voice reference from Google Drive: /content/drive/MyDrive/Chatterbox_Aussie_Turbo/voices/my_voice_reference.wav
Voice reference: 14.0s, 24000 Hz, 1 channel(s)


In [18]:
# 5 — Load actual Turbo model on GPU and condition it with your uploaded WAV voice.
import torch, numpy as np, hashlib
from chatterbox.tts_turbo import ChatterboxTurboTTS

MODEL = ChatterboxTurboTTS.from_pretrained(device=DEVICE)
_original_norm_loudness = MODEL.norm_loudness

def _norm_loudness_float32(wav, sr, target_lufs=-27):
    normalized = _original_norm_loudness(
        wav, sr, target_lufs=target_lufs
    )
    return np.asarray(normalized, dtype=np.float32)

MODEL.norm_loudness = _norm_loudness_float32
MODEL.prepare_conditionals(str(REFERENCE_WAV))
SAMPLE_RATE = int(MODEL.sr)
REFERENCE_HASH = hashlib.sha256(Path(REFERENCE_WAV).read_bytes()).hexdigest()
print('Loaded: Chatterbox Turbo on',torch.cuda.get_device_name(0))
print('Sample rate:',SAMPLE_RATE,'| reference fingerprint:',REFERENCE_HASH[:12])


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)


loaded PerthNet (Implicit) at step 250,000
Loaded: Chatterbox Turbo on Tesla T4
Sample rate: 24000 | reference fingerprint: 795d3dde29dc


In [19]:
# 5 — Render MP3 with FFmpeg. Short pilot first; no intermediate WAV left behind.
import numpy as np, tempfile, subprocess, json, time, hashlib
from IPython.display import Audio, display

SOUND_TAGS = {'clear throat','sigh','shush','cough','groan','sniff','gasp','chuckle','laugh'}
STYLE_TAGS = {'angry','fear','surprised','whispering','crying','happy',
              'sarcastic','dramatic','narration','advertisement'}
SUPPORTED_TAGS = SOUND_TAGS | STYLE_TAGS

def render_mp3(chunks, destination):
    """chunks: iterable of (speech, pause_after_seconds)."""
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    outfile = destination.with_name(destination.name+'.partial')
    frames = 0
    with tempfile.TemporaryDirectory() as tmpdir:
        pcm_wav = Path(tmpdir) / 'speech.wav'
        with sf.SoundFile(str(pcm_wav), mode='w', samplerate=SAMPLE_RATE, channels=1,
                          subtype='PCM_16', format='WAV') as wavfile:
            for n,(speech,pause) in enumerate(chunks, 1):
                print(f'  Speech chunk {n}: {speech[:90]}')
                with torch.inference_mode():
                    generated = MODEL.generate(text=speech)
                audio = generated.detach().float().cpu().numpy().reshape(-1)
                if audio.size == 0 or not np.isfinite(audio).all():
                    raise RuntimeError(f'Bad audio generated for chunk {n}.')
                audio = np.clip(audio, -1, 1)
                wavfile.write(audio)
                frames += audio.size
                if pause > 0:
                    padding = np.zeros(int(SAMPLE_RATE * pause), dtype=np.float32)
                    wavfile.write(padding)
                    frames += padding.size
        command = [FFMPEG,'-y','-hide_banner','-loglevel','error',
                   '-i',str(pcm_wav),'-codec:a','libmp3lame','-b:a',MP3_BITRATE,
                   '-f','mp3',str(outfile)]
        try:
            subprocess.run(command,check=True,capture_output=True,text=True)
            if not outfile.exists() or outfile.stat().st_size < 5_000:
                raise RuntimeError('MP3 output is missing or unexpectedly small.')
            outfile.replace(destination)
        finally:
            outfile.unlink(missing_ok=True)
    print(f'Created: {destination} | estimated duration: {frames/SAMPLE_RATE:.1f}s')
    return destination

PREVIEW_FILE = REPORT_DIR / 'aussie_male_voice_preview.mp3'
PREVIEW_META = PREVIEW_FILE.with_suffix('.json')
preview_fingerprint = hashlib.sha256((PREVIEW_TEXT + REFERENCE_HASH + 'turbo').encode()).hexdigest()
if RUN_VOICE_PREVIEW:
    previous_preview = {}
    if PREVIEW_META.exists():
        try: previous_preview = json.loads(PREVIEW_META.read_text(encoding='utf-8'))
        except ValueError: pass
    same_preview = PREVIEW_FILE.exists() and previous_preview.get('fingerprint') == preview_fingerprint
    if same_preview and not OVERWRITE_EXISTING:
        print('Using existing preview with matching text, model and voice.')
    else:
        render_mp3([(PREVIEW_TEXT, 0.15)], PREVIEW_FILE)
        PREVIEW_META.write_text(json.dumps({'fingerprint':preview_fingerprint},indent=2),encoding='utf-8')
    display(Audio(filename=str(PREVIEW_FILE)))
else:
    print('Voice preview disabled. Enable RUN_VOICE_PREVIEW to test the accent.')

Using existing preview with matching text, model and voice.


In [20]:
# 6 — DOCX parser: preserve narration and Turbo emotion tags, omit heading text.
# Controls such as [pause:2s] and [show original comment for 2-3 seconds] become silence.
import re, hashlib, csv
from docx import Document

RE_TAG = re.compile(r'\[([^\]]+)\]')
RE_PAUSE = re.compile(r'\[\s*pause\s*:\s*(\d+(?:\.\d+)?)(?:\s*[-–]\s*(\d+(?:\.\d+)?))?\s*(?:s|sec|secs|seconds?)?\s*\]',re.I)
RE_VISUAL = re.compile(r'\[\s*(?:pause\s+for|show\s+|photo\s+|image\s+|visual\s+)[^\]]*?(\d+(?:\.\d+)?)(?:\s*[-–]\s*(\d+(?:\.\d+)?))?\s*(?:s|sec|secs|seconds?)\s*\]',re.I)

def get_pause(m):
    first = float(m.group(1))
    second = float(m.group(2)) if m.group(2) else first
    if not (0 <= first <= second <= 30):
        raise ValueError('Pause duration must be between 0 and 30 seconds.')
    return (first+second)/2

def safe_chunks(txt, limit=MAX_CHARS):
    tokens = list(re.finditer(r'\[[^\]]+\]|\S+',txt))
    result=[]; start=0
    while start < len(tokens):
        end=start
        while end+1 < len(tokens) and tokens[end+1].end()-tokens[start].start() <= limit:
            end+=1
        if end == start and tokens[start].end()-tokens[start].start() > limit:
            raise ValueError('One unbroken token exceeds MAX_CHARS. Please add whitespace.')
        if end < len(tokens)-1:
            boundary=[i for i in range(start,end+1)
                      if re.search(r'[.!?…]["\'\u201d\u2019)]*$',tokens[i].group()) and
                      tokens[i].end()-tokens[start].start()>=limit*0.45]
            if boundary: end=boundary[-1]
        if end>start and end<len(tokens)-1:
            if tokens[end].group().lower().strip('[] ') in STYLE_TAGS and tokens[end].group().startswith('['):
                end-=1
        result.append(txt[tokens[start].start():tokens[end].end()])
        start=end+1
    return result

def parse_docx(docpath):
    doc=Document(str(docpath))
    paras=[]
    for p in doc.paragraphs:
        txt=p.text.strip()
        style=p.style.name if p.style else ''
        if txt and not style.startswith('Heading') and style not in ('Title','Subtitle','Caption','Header','Footer'):
            paras.append(txt)
    if not paras:
        raise ValueError('No narrative paragraphs found. Story text must not be formatted as a heading.')
    final=[]
    def add(s, after):
        chunks=safe_chunks(s)
        for i,c in enumerate(chunks):
            final.append((c, after if i==len(chunks)-1 else CHUNK_PAUSE_SECONDS))
    for para in paras:
        for bracket in RE_TAG.finditer(para):
            raw=bracket.group(1).strip().lower()
            if raw not in SUPPORTED_TAGS and not (RE_PAUSE.fullmatch(bracket.group()) or RE_VISUAL.fullmatch(bracket.group())):
                raise ValueError(f'Unknown [tag] in {docpath.name}: [{raw}]. Fix typo or put stage direction outside script.')
        normalized=RE_VISUAL.sub(lambda m: f'[pause:{get_pause(m):g}s]',para)
        pos=0
        for match in RE_PAUSE.finditer(normalized):
            spoken=normalized[pos:match.start()].strip()
            if spoken: add(spoken,get_pause(match))
            elif final:
                last,pause=final[-1]; final[-1]=(last,pause+get_pause(match))
            pos=match.end()
        spoken=normalized[pos:].strip()
        if spoken: add(spoken,PARAGRAPH_PAUSE_SECONDS)
    if not final:
        raise ValueError('No words remain after removing stage directions.')
    return final

print('DOCX input directory:',DOCX_DIR.resolve())
files_found=sorted([p for p in DOCX_DIR.rglob('*') if p.is_file() and p.suffix.lower()=='.docx' and not p.name.startswith('~$')],key=lambda p:str(p).casefold())
print('DOCX files currently found:',len(files_found))
for item in files_found[:5]: print(' •',item.relative_to(DOCX_DIR))

DOCX input directory: /content/drive/MyDrive/Chatterbox_Aussie_Turbo/input_docx
DOCX files currently found: 1
 • Narratorstory.docx


In [21]:
# 7 — Optional sequential DOCX → MP3 batch. OFF until the male Aussie voice passes your preview.
# Outputs go to a separate Australian folder; already-generated files are not overwritten.
if not RUN_DOCX_BATCH:
    print('DOCX batch is disabled for accent testing. If the voice sounds good, set RUN_DOCX_BATCH=True in Cell 1.')
else:
    doc_paths = sorted([p for p in DOCX_DIR.rglob('*') if p.is_file() and p.suffix.lower()=='.docx'
                        and not p.name.startswith('~$')],key=lambda p:str(p).casefold())
    if not doc_paths:
        print('No DOCX files yet. Add files to:',DOCX_DIR.resolve())
    else:
        tasks = doc_paths if RUN_LIMIT is None else doc_paths[:RUN_LIMIT]
        log=[]
        for index,path in enumerate(tasks,1):
            target = OUT_DIR / path.relative_to(DOCX_DIR).with_suffix('.mp3')
            target.parent.mkdir(parents=True,exist_ok=True)
            fingerprint = hashlib.sha256(path.read_bytes()+REFERENCE_HASH.encode()+
                b'turbo').hexdigest()
            metafile = target.with_suffix('.json')
            existing = {}
            if metafile.exists():
                try: existing=json.loads(metafile.read_text(encoding='utf-8'))
                except ValueError: pass
            print(f'[{index}/{len(tasks)}] {path.relative_to(DOCX_DIR)}')
            try:
                if target.exists() and not OVERWRITE_EXISTING:
                    if existing.get('fingerprint') == fingerprint:
                        print('   SKIP: already generated from the same input and voice.')
                        status='SKIPPED'
                    else:
                        print('   REVIEW: MP3 exists but settings/input changed; set OVERWRITE_EXISTING=True to regenerate.')
                        status='REVIEW_EXISTING'
                else:
                    chunks=parse_docx(path)
                    render_mp3(chunks,target)
                    metafile.write_text(json.dumps({'source':str(path.relative_to(DOCX_DIR)),
                         'fingerprint':fingerprint,'model':'turbo',
                         'reference_sha256':REFERENCE_HASH,'chunk_count':len(chunks)},indent=2),encoding='utf-8')
                    status='CREATED'
                log.append({'story':str(path.relative_to(DOCX_DIR)),'status':status,'detail':''})
            except Exception as exc:
                print('   ERROR:',type(exc).__name__,str(exc))
                log.append({'story':str(path.relative_to(DOCX_DIR)),
                            'status':'ERROR','detail':f'{type(exc).__name__}: {exc}'})
        logfile=REPORT_DIR / 'batch_results.csv'
        with logfile.open('w',newline='',encoding='utf-8-sig') as fh:
            writer=csv.DictWriter(fh,fieldnames=['story','status','detail'])
            writer.writeheader();writer.writerows(log)
        print('Batch summary:',{status:sum(r['status']==status for r in log)
                               for status in sorted(set(r['status'] for r in log))})
        print('MP3 output:',OUT_DIR.resolve())
        print('Processing report:',logfile.resolve())

[1/1] Narratorstory.docx
  Speech chunk 1: This one's a user story! THANK YOU and keep posting such stories in comments and They will


 18%|█▊        | 182/1000 [00:03<00:15, 51.97it/s]


S3 Token -> Mel Inference...


100%|██████████| 2/2 [00:00<00:00, 11.48it/s]


  Speech chunk 2: A pub owner, some roadworks, and a brewery manager who really should've known better.


...

  Speech chunk 37: [laugh]


  2%|▎         | 25/1000 [00:00<00:18, 52.75it/s]


S3 Token -> Mel Inference...


100%|██████████| 2/2 [00:00<00:00, 15.77it/s]


Created: /content/drive/MyDrive/Chatterbox_Aussie_Turbo/output_mp3_aussie_male/Narratorstory.mp3 | estimated duration: 156.2s
Batch summary: {'CREATED': 1}
MP3 output: /content/drive/MyDrive/Chatterbox_Aussie_Turbo/output_mp3_aussie_male
Processing report: /content/drive/MyDrive/Chatterbox_Aussie_Turbo/reports_aussie_male/batch_results.csv


### How to use the Drive workspace

1. Open this notebook in **Google Colab** and enable a **T4 GPU** (or better) under Runtime settings.
2. Run all cells. Approve Drive mount. The notebook creates the workspace in `MyDrive/Chatterbox_Aussie_Turbo/`.
3. When prompted, upload your own `.wav` voice reference. Then listen to the preview at `reports_aussie_male/aussie_male_voice_preview.mp3`.
4. Upload `.docx` scripts to the **Drive** folder `input_docx/`. Keep using the same folder between Colab sessions.
5. In the settings cell, set `RUN_DOCX_BATCH=True` and retain `RUN_LIMIT=1` for the first complete story. Rerun **all cells in order** (or the settings cell and DOCX parsing/batch cells if model is still loaded).
6. After verifying, set `RUN_LIMIT=None` to process every DOCX. Existing outputs will be skipped unless `OVERWRITE_EXISTING=True`; outputs land under `output_mp3_aussie_male/`.

**Notes:** Colab GPU availability and runtime limits depend on your plan. When the reference WAV is missing, the notebook asks you to upload it and stores it in Drive for reuse. The official Chatterbox code has separate Nano and Turbo models; this notebook explicitly loads **Turbo** with `nano=False`. Only use reference audio you have permission to use. This notebook has not been executed inside an actual Colab GPU runtime here.
